In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [ ]:
df = pd.read_csv('jamb_exam_results.csv')
df.columns = df.columns.str.lower().str.replace(' ', '_')

In [ ]:
if 'student_id' in df.columns:
    df = df.drop('student_id', axis=1)

df = df.fillna(0)

X = df.drop('jamb_score', axis=1)
y = df['jamb_score']

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=1
)

In [ ]:
dv = DictVectorizer(sparse=True)

train_dict = X_train.to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = X_val.to_dict(orient='records')
X_val = dv.transform(val_dict)

test_dict = X_test.to_dict(orient='records')
X_test = dv.transform(test_dict)

feature_names = dv.get_feature_names_out()

In [ ]:
dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)

tree_feature_index = dt.tree_.feature[0]
feature_used = feature_names[tree_feature_index]
print(f"Вопрос 1. Признак, используемый для разбиения: {feature_used}")

Вопрос 1. Признак, используемый для разбиения: study_hours_per_week


In [ ]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_val = rf.predict(X_val)
rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
print(f"Вопрос 2. RMSE на валидации: {rmse_val:.3f}")

Вопрос 2. RMSE на валидации: 42.137


In [ ]:
best_rmse = float('inf')
best_n = 0
n_estimators_range = range(10, 201, 10)
rmse_values = []

for n in n_estimators_range:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred_val = rf.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
    rmse_values.append(rmse)
    if rmse < best_rmse:
        best_rmse = rmse
        best_n = n

for i in range(1, len(rmse_values)):
    improvement = rmse_values[i-1] - rmse_values[i]
    if improvement < 0.001:
        stopping_point = n_estimators_range[i-1]
        break
else:
    stopping_point = best_n

print(f"Вопрос 3. RMSE перестает улучшаться после n_estimators={stopping_point}")

Вопрос 3. RMSE перестает улучшаться после n_estimators=90


In [31]:
max_depth_values = [10, 15, 20, 25]
best_depth = None
best_rmse = float('inf')

n_estimators_used = stopping_point

for depth in max_depth_values:
    rf = RandomForestRegressor(n_estimators=n_estimators_used,
                               max_depth=depth,
                               random_state=1,
                               n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred_val = rf.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
    print(f"max_depth={depth}, RMSE={rmse:.3f}")
    if rmse < best_rmse:
        best_rmse = rmse
        best_depth = depth

print(f"Лучшее max_depth: {best_depth}")

max_depth=10, RMSE=40.174
max_depth=15, RMSE=40.497
max_depth=20, RMSE=40.493
max_depth=25, RMSE=40.513
Лучшее max_depth: 10


In [32]:
rf_final = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf_final.fit(X_train, y_train)

feature_importances = rf_final.feature_importances_
feature_importance_dict = dict(zip(feature_names, feature_importances))

# Целевые признаки
target_features = ['study_hours_per_week', 'attendance_rate', 'distance_to_school', 'teacher_quality']

most_important_target = max(
    target_features, key=lambda x: feature_importance_dict.get(x, 0)
)
print(f"Вопрос 5. Самый важный признак: {most_important_target}")

Вопрос 5. Самый важный признак: study_hours_per_week
